In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat
import warnings
warnings.filterwarnings('ignore')

from imports import *
from config import dir_config

In [4]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)
output_folder_name = 'equal_block_cross_validation_1coh_50choice'

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"], session_to_exclude)]

neuron_metadata = pd.read_csv(Path(processed_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"], session_to_exclude)].reset_index()


In [5]:
def extract_neuron_data(neuron_id, prior_cond, outcome_filter = "correct_only"):
    session_name = neuron_metadata.loc[neuron_metadata["neuron_id"] == neuron_id, "session_id"].values[0]
    data_path = Path(compiled_dir, session_name)

    # Load neural and behavioral data
    try:
        spike_times = np.load(data_path / "spike_times.npy")
        spike_clusters = np.load(data_path / "spike_clusters.npy")
    except:
        spike_times = loadmat(Path(compiled_dir, session_name, "spike_times.mat"))
        spike_times = spike_times["spike_times"][0]
        spike_clusters = loadmat(Path(compiled_dir, session_name, "spike_clusters.mat"))
        spike_clusters = spike_clusters["spike_clusters"][0]

    # Get neuron spike times
    cluster_id = neuron_metadata.cluster[neuron_metadata["neuron_id"] == neuron_id].values[0]
    neuron_spike_times = spike_times[spike_clusters == cluster_id]
    neuron_spike_times = (neuron_spike_times / 30).round().astype(int)  # Convert to ms

    # Get timestamps and trial data
    timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps.csv"), index_col=None)
    trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial.csv"), index_col=None)

    # Process trial data
    GP_trial_data = trial_info[trial_info.task_type == 1].reset_index(drop=True)
    # signed coherence
    GP_trial_data["signed_coherence"] = GP_trial_data["coherence"] * (2*GP_trial_data["target"]-1)

    GP_trial_data = GP_trial_data[GP_trial_data.reaction_time.notna()]
    # include equal block only
    if prior_cond == "equal_only":
        GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF == 50]
        GP_trial_data["state"] = 0
    elif prior_cond == "unequal_only":
        GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF != 50]
        GP_trial_data["state"] = 0
    else:
        raise NotImplementedError("Multiple blocks not implemented yet.")

    coh_levels = np.sort(GP_trial_data['signed_coherence'].unique()) /100  # Normalize coherence

    if outcome_filter == "correct_only":
        GP_trial_data = GP_trial_data[GP_trial_data.outcome == 1].reset_index()
    elif outcome_filter == "incorrect_only":
        GP_trial_data = GP_trial_data[GP_trial_data.outcome == 0].reset_index()

    return GP_trial_data, neuron_spike_times, timestamps, coh_levels


def create_neuroglm_trials(session_data, timestamps, neuron_spike_times, bin_size=1.0):
    """
    Create trial structure following neuroGLM format.
    """
    trials = []

    # Convert timestamps to ms
    timestamps_ms = (timestamps / 30).round()

    for idx, row in session_data.iterrows():
        trial_idx = row.trial_number - 1  # Convert to 0-based

        # Trial timing (relative to target onset - 50ms)
        target_onset = timestamps_ms.loc[trial_idx, "target_onset"]
        trial_start = target_onset - 50
        trial_end = timestamps_ms.loc[trial_idx, "response_onset"]
        duration = trial_end - trial_start # -50ms of target onset to response onset

        if pd.isna(duration) or duration <= 0:
            continue

        # Get trial spike times (relative to trial start)
        trial_spikes = neuron_spike_times[
            (neuron_spike_times >= trial_start) &
            (neuron_spike_times <= trial_end)
        ] - trial_start

        # Create binned spike train
        n_bins = int(np.ceil(duration / bin_size))
        spike_train = np.zeros(n_bins)

        for spike_time in trial_spikes:
            bin_idx = int(np.floor(spike_time / bin_size))
            if 0 <= bin_idx < n_bins:
                spike_train[bin_idx] += 1

        # Event timings (relative to trial start)
        events = {
            'target_onset': 50,  # Always 50ms into trial
            'stimulus_onset': timestamps_ms.loc[trial_idx, "stimulus_onset"] - trial_start,
            'stimulus_offset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
            'response_onset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
        }

        # Trial structure
        trial = {
            'duration': duration,
            'spike_train': spike_train,
            'n_bins': n_bins,

            # Event timings
            'target_onset': events['target_onset'],
            'stimulus_onset': events['stimulus_onset'],
            'stimulus_offset': events['stimulus_offset'],
            'response_onset': events['response_onset'],

            # Experimental variables
            'coherence': row.signed_coherence / 100,  # Normalize coherence
            'choice': row.choice,
            'state': row.state,
            'reaction_time': row.reaction_time,

            # Trial metadata
            'trial_idx': int(trial_idx),
        }

        trials.append(trial)

    return pd.DataFrame(trials)

## Load and prepare data

In [ ]:
for prior_cond in ["equal_only", "unequal_only"]:
    for outcome_filter in ["correct_only", "all"]:

        # create folder for storing results if it doesn't exist
        output_dir = Path(processed_dir, 'poisson_glm', "data", f"prior_cond_{prior_cond}_outcome_{outcome_filter}")
        output_dir.mkdir(parents=True, exist_ok=True)

        for neuron_id in neuron_metadata.neuron_id.unique():
            session_data, neuron_spike_times, timestamps, coh_levels = extract_neuron_data(neuron_id, prior_cond=prior_cond, outcome_filter=outcome_filter)
            trials_df = create_neuroglm_trials(session_data, timestamps, neuron_spike_times)
            # save trials_df for later use
            trials_df.to_parquet(output_dir / f"{neuron_id}.parquet", index=False)
